# MIntRec2.0 — LOCAL frozen-teacher extraction (4-bit, sub-sampled frames)

Run the Qwen2.5-Omni-3B teacher **locally on a small GPU** (tested on a 6.4 GB
RTX 3060 Laptop) by feeding **only a few sub-sampled video frames** instead of the
whole clip. Same design as the production `src/mintrec/teacher_probe/extract_features.py`,
but the video path is replaced by cv2 frame sampling (the MOSEI-notebook trick).

**Verified on a 6.4 GB 3060**: 4-bit load = 3.8 GB; forward with 4 frames @336px +
4 s audio = **5.16 GB peak, ~2.3 s/sample**. So it fits; the full-clip default
(many frames) would OOM.

Pipeline (kept identical to the production extractor except frame sampling):
`prompt_first` -> content order **[text(prompt+transcript), video-frames, audio]**,
audio **last**; `use_audio_in_video=False`; pool **audio_mean** at layers
`[24,27,30,34]` (+ mean).

> NOTE: the raw MIntRec2.0 videos must be placed at `data/mintrec/MIntRec2.0/`
> (download from thuiar). Without them, only the **synthetic VRAM smoke test**
> (cell 4) runs — that alone confirms local feasibility.

> These local **4-bit + sub-sampled** features are a *different* extraction than the
> existing RunPod **bf16 + full-video** ones (`teacher_features/...bf16...`); don't
> mix the two in one probe.

In [ ]:
# ── config ────────────────────────────────────────────────────────────────
import sys, os, gc, json
from pathlib import Path
import numpy as np

if sys.platform == "win32":
    for _p in os.environ.get("PATH", "").split(";"):
        if _p and os.path.exists(os.path.join(_p, "avcodec-62.dll")):
            os.add_dll_directory(_p); break

_cands = [Path.cwd(), *Path.cwd().parents]
_SRC = next((p for p in _cands if p.name == "src"), None)
if _SRC is None:
    _SRC = next((p / "src" for p in _cands if (p / "src" / "common" / "config.py").exists()), None)
assert _SRC is not None, "cannot locate src/"
if str(_SRC) not in sys.path: sys.path.insert(0, str(_SRC))
from common.config import MINTREC_DATA   # noqa: E402

MODEL_NAME = "Qwen/Qwen2.5-Omni-3B"
DTYPE      = "4bit"            # local default; bf16 needs a big GPU
LLM_LAYERS = [24, 27, 30, 34]
MEAN_COMBO = "L24-27-30-34"
ADD_GEN_PROMPT = True

NUM_VIDEO_FRAMES = 4          # MUST be even (Qwen temporal patch=2). 2 if VRAM-tight, 6-8 if you have room.
FRAME_MAX_SIDE   = 336        # resize longest side -> caps vision tokens (~27/frame)
AUDIO_SR = 16000

AUDIO_START_ID_DEFAULT = 151647
AUDIO_END_ID_DEFAULT   = 151648

ANNO_DIR  = MINTREC_DATA / "MIntRec2.0"      # train/dev/test.tsv + video/ live here
VIDEO_DIR = ANNO_DIR / "video"
DTAG = f"mintrec2.0__qwen2.5-omni-3b-{DTYPE}__pf_text-video{NUM_VIDEO_FRAMES}f-audio__audiomean"
OUT_DIR = MINTREC_DATA / "teacher_features" / DTAG

TASK_PROMPT_TEMPLATE = (
    "You are analyzing a short TV-show clip to recognize the speaker's intent "
    "among 30 fine-grained intent classes. "
    'The transcript of what is said is: "{text}".'
)
print("ANNO_DIR:", ANNO_DIR, "| exists:", ANNO_DIR.exists())
print("DTAG    :", DTAG)

In [ ]:
# ── frame sampling (cv2) + audio load ─────────────────────────────────────
import cv2, librosa
from PIL import Image

def _resize_max_side(img, ms):
    h, w = img.shape[:2]; sc = ms / max(h, w)
    if sc < 1.0:
        img = cv2.resize(img, (int(round(w*sc)), int(round(h*sc))), interpolation=cv2.INTER_AREA)
    return img

def extract_frames(path, n_frames=None, max_side=None):
    # uniformly sample n interior frames across the whole clip -> list[PIL.Image] (RGB)
    n_frames = n_frames or NUM_VIDEO_FRAMES; max_side = max_side or FRAME_MAX_SIDE
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened(): raise FileNotFoundError(f"cannot open {path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    end = (total/fps) if total else 1.0
    times = np.linspace(0.0, end, n_frames + 2)[1:-1]
    frames = []
    for t in times:
        idx = int(round(t*fps));  idx = min(idx, total-1) if total else 0
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, fr = cap.read()
        if not ok: continue
        frames.append(Image.fromarray(_resize_max_side(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB), max_side)))
    cap.release()
    if not frames: raise RuntimeError(f"no frames from {path}")
    if len(frames) % 2 == 1:                # Qwen needs an even frame count
        frames = frames[:-1] if len(frames) > 1 else frames + frames
    return frames

def load_audio(path):
    wav, _ = librosa.load(str(path), sr=AUDIO_SR, mono=True)   # decode audio from the mp4
    return wav
print("frame/audio helpers ready")

In [ ]:
# ── load 4-bit teacher + extraction helpers (mirrors the production extractor)
import torch
from transformers import BitsAndBytesConfig, Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

def load_teacher(dtype="4bit"):
    kw = dict(device_map="auto", attn_implementation="sdpa")
    if dtype == "4bit":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    else:
        kw["torch_dtype"] = torch.bfloat16
    proc = Qwen2_5OmniProcessor.from_pretrained(MODEL_NAME)
    mdl = Qwen2_5OmniForConditionalGeneration.from_pretrained(MODEL_NAME, **kw); mdl.eval()
    return mdl, proc

def get_special_id(model, name, default):
    cfg = model.config
    if hasattr(cfg, name): return getattr(cfg, name)
    for sub in ("thinker_config", "talker_config"):
        if hasattr(cfg, sub) and hasattr(getattr(cfg, sub), name):
            return getattr(getattr(cfg, sub), name)
    return default

def find_audio_indices(input_ids, s_id, e_id):
    ids = input_ids[0].tolist(); s, e = ids.index(s_id), ids.index(e_id)
    idx = list(range(s+1, e))
    if not idx: raise ValueError("no audio tokens between markers")
    return idx, s, e

def _cpu16(t): return t.detach().float().cpu().to(torch.float16)

def pool_audio_mean(hs, audio_idx):
    feats, per = {}, []
    for L in LLM_LAYERS:
        v = hs[L][0][audio_idx].mean(0); feats[f"pf_audio_mean_l{L}"] = _cpu16(v); per.append(v)
    feats[f"pf_audio_mean_{MEAN_COMBO}"] = _cpu16(torch.stack(per, 0).mean(0))
    return feats

def build_inputs(proc, frames, wav, transcript):
    conv = [{"role":"user","content":[
        {"type":"text","text":TASK_PROMPT_TEMPLATE.format(text=transcript)},
        {"type":"video","video":frames},   # list[PIL] = pre-sampled frames
        {"type":"audio","audio":wav}]}]
    txt = proc.apply_chat_template(conv, add_generation_prompt=ADD_GEN_PROMPT, tokenize=False)
    a, i, v = process_mm_info(conv, use_audio_in_video=False)
    return proc(text=txt, audio=a, images=i, videos=v, return_tensors="pt",
                padding=True, use_audio_in_video=False).to(model.device)

@torch.no_grad()
def extract_sample(model, proc, frames, wav, transcript, a0, a1):
    inp = build_inputs(proc, frames, wav, transcript)
    out = model.thinker(**inp, output_hidden_states=True, return_dict=True)
    audio_idx, s, e = find_audio_indices(inp["input_ids"], a0, a1)
    feats = pool_audio_mean(out.hidden_states, audio_idx)
    return feats, {"seq_len": int(inp["input_ids"].shape[1]), "num_audio_tokens": len(audio_idx)}

print("loading 4-bit teacher ...")
model, processor = load_teacher(DTYPE)
A_START = get_special_id(model, "audio_start_token_id", AUDIO_START_ID_DEFAULT)
A_END   = get_special_id(model, "audio_end_token_id", AUDIO_END_ID_DEFAULT)
print("loaded | VRAM:", round(torch.cuda.memory_allocated()/1e9,2), "GB | audio ids", A_START, A_END)

In [ ]:
# ── SYNTHETIC VRAM smoke test (NO DATA NEEDED) — proves it fits on your GPU ──
import torch, time
def _synth(n_frames, secs, side=FRAME_MAX_SIDE):
    frames=[Image.fromarray(np.random.randint(0,255,(side,side,3),dtype=np.uint8)) for _ in range(n_frames)]
    wav=(0.1*np.sin(2*np.pi*220*np.arange(int(AUDIO_SR*secs))/AUDIO_SR)).astype(np.float32)
    return frames, wav

for nf in [NUM_VIDEO_FRAMES]:
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    fr, wav = _synth(nf, 4.0); t=time.time()
    feats, meta = extract_sample(model, processor, fr, wav, "I really need this done now.", A_START, A_END)
    peak = torch.cuda.max_memory_allocated()/1e9 if torch.cuda.is_available() else 0
    print(f"frames={nf}: seq={meta['seq_len']} audio_toks={meta['num_audio_tokens']} "
          f"fwd={time.time()-t:.2f}s peakVRAM={peak:.2f}GB dim={feats[f'pf_audio_mean_{MEAN_COMBO}'].shape[0]}")
print("If this printed without OOM, local extraction is feasible on this GPU.")

In [ ]:
# ── MIntRec index (needs raw data at ANNO_DIR) ────────────────────────────
import pandas as pd

def load_split_df(name):
    df = pd.read_csv(ANNO_DIR / f"{name}.tsv", sep="\t", dtype=str, keep_default_na=False)
    df.columns = [c.strip() for c in df.columns]
    df["id"] = df["id"].str.strip(); df["text"] = df["text"].str.strip()
    df["label"] = df["label"].str.strip().str.lower()
    df["dia"] = df["id"].str.split("_").str[0]; df["utt"] = df["id"].str.split("_").str[1]
    return df[df["label"] != ""].reset_index(drop=True)

def build_label2id(dfs):
    labels = sorted({l for df in dfs for l in df["label"].unique()})
    return {l: i for i, l in enumerate(labels)}

def build_stem2path():
    mp4 = list(VIDEO_DIR.rglob("*.mp4")); return {p.stem: p for p in mp4}, len(mp4)

def find_video(row, s2p):
    for k in (f"MIntRec2.0_{row['id']}", row["id"], f"dia{row['dia']}_utt{row['utt']}"):
        if k in s2p: return s2p[k]
    return None

assert ANNO_DIR.exists(), f"raw MIntRec2.0 not found at {ANNO_DIR} — download it first"
dfs = {s: load_split_df(s) for s in ("train", "dev") if (ANNO_DIR / f"{s}.tsv").exists()}
label2id = build_label2id(dfs.values())
stem2path, n_mp4 = build_stem2path()
df = dfs.get("dev", next(iter(dfs.values())))
print(f"{ {k: len(v) for k, v in dfs.items()} } | {len(label2id)} intents | {n_mp4} mp4 indexed")
df.head()

In [ ]:
# ── VISUAL CHECK: sampled frames for one utterance ────────────────────────
import matplotlib.pyplot as plt
r = None
for _, c in df.iterrows():
    if find_video(c, stem2path) is not None: r = c; break
assert r is not None, "no row matched an mp4 — check find_video patterns / VIDEO_DIR"
vp = find_video(r, stem2path); frames = extract_frames(vp); wav = load_audio(vp)
print(r["id"], "| label:", r["label"], "| frames:", len(frames), frames[0].size, "| audio %.2fs" % (len(wav)/AUDIO_SR))
fig, ax = plt.subplots(1, len(frames), figsize=(4*len(frames), 4))
for a, im in zip(np.atleast_1d(ax), frames): a.imshow(im); a.axis("off")
plt.suptitle(f"{r['id']}  {r['label']}  | {r['text'][:70]}"); plt.tight_layout(); plt.show()

In [ ]:
# ── REAL smoke test: 5 utterances (needs data + model) ────────────────────
import torch
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
shown = 0
for _, r in df.iterrows():
    vp = find_video(r, stem2path)
    if vp is None: continue
    try:
        frames = extract_frames(vp); wav = load_audio(vp)
        if wav.size == 0: continue
        feats, meta = extract_sample(model, processor, frames, wav, r["text"], A_START, A_END)
        print(f"[{shown}] {r['id']:<16} {r['label']:<18} seq={meta['seq_len']:<4} audio_toks={meta['num_audio_tokens']}")
    except Exception as ex:
        print(f"[{shown}] {r['id']} FAILED: {type(ex).__name__}: {ex}")
    shown += 1
    if shown >= 5: break
if torch.cuda.is_available(): print("peak VRAM:", round(torch.cuda.max_memory_allocated()/1e9,2), "GB")

In [ ]:
# ── MINI extraction: first LIMIT utterances -> smoke .pt ──────────────────
from tqdm.auto import tqdm
import torch
LIMIT = 20
fb, labels, ids, meta_all = {}, [], [], []
done = 0
for _, r in tqdm(df.iterrows(), total=min(LIMIT, len(df)), desc="extract"):
    if done >= LIMIT: break
    vp = find_video(r, stem2path)
    if vp is None: continue
    try:
        frames = extract_frames(vp); wav = load_audio(vp)
        if wav.size == 0: continue
        feats, meta = extract_sample(model, processor, frames, wav, r["text"], A_START, A_END)
    except Exception as ex:
        print("skip", r["id"], ex); continue
    for k, v in feats.items(): fb.setdefault(k, []).append(v)
    labels.append(label2id[r["label"]]); ids.append(r["id"])
    meta.update({"id": r["id"], "label": r["label"], "label_id": label2id[r["label"]]}); meta_all.append(meta)
    done += 1
    if done % 5 == 0 and torch.cuda.is_available(): torch.cuda.empty_cache(); gc.collect()

result = {"labels": torch.tensor(labels, dtype=torch.long), "sample_ids": ids, "metadata": meta_all,
          "features": {k: torch.stack(v, 0) for k, v in fb.items()},
          "feature_dim": (next(iter(fb.values()))[0].shape[0] if fb else 0)}
OUT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(result, OUT_DIR / "smoke_features.pt")
print(f"saved {len(ids)} samples x {len(fb)} features (dim={result['feature_dim']}) -> {OUT_DIR/'smoke_features.pt'}")

## Next steps

- If the **synthetic** cell printed peak VRAM < your card and the **real** smoke test
  located audio tokens, you can extract MIntRec2.0 locally at 4-bit with sub-sampled
  frames.
- Tune `NUM_VIDEO_FRAMES` (2 / 4 / 6-8) by VRAM headroom; this is also a thesis
  ablation axis (teacher dense vs student sparse = privileged distillation).
- To run the **full** corpus, port this loop into a sharded/resume-safe
  `extract_features.py` like `src/mintrec/teacher_probe/extract_features.py`
  (shard flush / `_load_shards` / `_count_done`), then probe the pooled features.